In [1]:
from collections import defaultdict
from itertools import product

class HMMPOSTagger:
    def __init__(self):
        # Store counts for probability calculations
        self.tag_counts = defaultdict(int)
        self.initial_tag_counts = defaultdict(int)
        self.transition_counts = defaultdict(lambda: defaultdict(int))
        self.emission_counts = defaultdict(lambda: defaultdict(int))
        
        # Store all unique tags and words
        self.all_tags = set()
        self.all_words = set()
        
        # Smoothing parameter
        self.smoothing = 1
    
    def train(self, training_data):
        print("Training the model...")
        
        # Count everything
        for sentence in training_data:
            # First tag in sentence (for initial probabilities)
            first_word, first_tag = sentence[0]
            self.initial_tag_counts[first_tag] += 1
            
            # Process each word-tag pair
            for i, (word, tag) in enumerate(sentence):
                self.all_tags.add(tag)
                self.all_words.add(word)
                self.tag_counts[tag] += 1
                self.emission_counts[tag][word] += 1
                
                # Transition: previous tag -> current tag
                if i > 0:
                    prev_tag = sentence[i-1][1]
                    self.transition_counts[prev_tag][tag] += 1
        
        print(f"Found {len(self.all_tags)} unique tags: {sorted(self.all_tags)}")
        print(f"Found {len(self.all_words)} unique words")
        print("Training complete!\\n")
    
    def get_initial_prob(self, tag):
        total_sentences = sum(self.initial_tag_counts.values())
        count = self.initial_tag_counts[tag]
        # Add-one smoothing
        return (count + self.smoothing) / (total_sentences + self.smoothing * len(self.all_tags))
    
    def get_transition_prob(self, prev_tag, curr_tag):
        prev_tag_count = self.tag_counts[prev_tag]
        transition_count = self.transition_counts[prev_tag][curr_tag]
        # Add-one smoothing
        return (transition_count + self.smoothing) / (prev_tag_count + self.smoothing * len(self.all_tags))
    
    def get_emission_prob(self, tag, word):
        tag_count = self.tag_counts[tag]
        emission_count = self.emission_counts[tag][word]
        # Add-one smoothing
        vocab_size = len(self.all_words) + 1  # +1 for unknown words
        return (emission_count + self.smoothing) / (tag_count + self.smoothing * vocab_size)
    
    def calculate_sequence_prob(self, sentence, tag_sequence):
        prob = 1.0
        
        for i, word in enumerate(sentence):
            tag = tag_sequence[i]
            
            if i == 0:
                # Initial probability × emission probability
                prob *= self.get_initial_prob(tag)
            else:
                # Transition probability
                prev_tag = tag_sequence[i-1]
                prob *= self.get_transition_prob(prev_tag, tag)
            
            # Emission probability
            prob *= self.get_emission_prob(tag, word)
        
        return prob
    
    def brute_force_decode(self, sentence):

        print(f"Decoding: {' '.join(sentence)}")
        
        n = len(sentence)
        tags = list(self.all_tags)
        
        # Generate all possible tag sequences
        # For 3 words and 5 tags: 5^3 = 125 sequences!
        all_sequences = product(tags, repeat=n)
        
        best_sequence = None
        best_prob = 0
        
        sequence_count = 0
        for tag_sequence in all_sequences:
            sequence_count += 1
            prob = self.calculate_sequence_prob(sentence, tag_sequence)
            
            if prob > best_prob:
                best_prob = prob
                best_sequence = tag_sequence
        
        print(f"  Checked {sequence_count} possible sequences")
        print(f"  Best probability: {best_prob:.2e}")
        print(f"  Best tags: {list(best_sequence)}\\n")
        
        return list(best_sequence)

In [2]:
training_data = [
    [("The", "DET"), ("dog", "NOUN"), ("barks", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("meows", "VERB")],
    [("A", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("meows", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("cat", "NOUN"), ("runs", "VERB")],
    [("A", "DET"), ("big", "ADJ"), ("dog", "NOUN"), ("sleeps", "VERB")],
    [("The", "DET"), ("small", "ADJ"), ("cat", "NOUN"), ("runs", "VERB")],
    [("Dogs", "NOUN"), ("bark", "VERB")],
    [("Cats", "NOUN"), ("meow", "VERB")],
    [("The", "DET"), ("dog", "NOUN"), ("in", "ADP"), ("the", "DET"), ("house", "NOUN"), ("barks", "VERB")],
    [("A", "DET"), ("cat", "NOUN"), ("on", "ADP"), ("the", "DET"), ("mat", "NOUN"), ("sleeps", "VERB")],
]

In [3]:
tagger = HMMPOSTagger()
tagger.train(training_data)

Training the model...
Found 5 unique tags: ['ADJ', 'ADP', 'DET', 'NOUN', 'VERB']
Found 19 unique words
Training complete!\n


In [4]:
test_sentences = [
    ["The", "dog", "barks"],
    ["A", "cat", "sleeps"],
    ["The", "big", "dog", "runs"],
    ["Dogs", "bark"],
]
print("TESTING THE TAGGER")

for sentence in test_sentences:
    predicted_tags = tagger.brute_force_decode(sentence)
    
    # Show the result nicely
    print("Result:")
    for word, tag in zip(sentence, predicted_tags):
        print(f"  {word:10} -> {tag}")
    print()

TESTING THE TAGGER
Decoding: The dog barks
  Checked 125 possible sequences
  Best probability: 1.94e-03
  Best tags: ['DET', 'NOUN', 'VERB']\n
Result:
  The        -> DET
  dog        -> NOUN
  barks      -> VERB

Decoding: A cat sleeps
  Checked 125 possible sequences
  Best probability: 1.42e-03
  Best tags: ['DET', 'NOUN', 'VERB']\n
Result:
  A          -> DET
  cat        -> NOUN
  sleeps     -> VERB

Decoding: The big dog runs
  Checked 625 possible sequences
  Best probability: 7.02e-05
  Best tags: ['DET', 'ADJ', 'NOUN', 'VERB']\n
Result:
  The        -> DET
  big        -> ADJ
  dog        -> NOUN
  runs       -> VERB

Decoding: Dogs bark
  Checked 25 possible sequences
  Best probability: 3.37e-04
  Best tags: ['NOUN', 'VERB']\n
Result:
  Dogs       -> NOUN
  bark       -> VERB



In [6]:
print("EXAMPLE PROBABILITIES")

print("Initial Probabilities (first word in sentence):")
for tag in sorted(tagger.all_tags):
    prob = tagger.get_initial_prob(tag)
    print(f"  P({tag}) = {prob:.4f}")

print("\\nSome Transition Probabilities:")
print(f"  P(NOUN | DET) = {tagger.get_transition_prob('DET', 'NOUN'):.4f}")
print(f"  P(VERB | NOUN) = {tagger.get_transition_prob('NOUN', 'VERB'):.4f}")
print(f"  P(ADJ | DET) = {tagger.get_transition_prob('DET', 'ADJ'):.4f}")

print("\\nSome Emission Probabilities:")
print(f"  P('dog' | NOUN) = {tagger.get_emission_prob('NOUN', 'dog'):.4f}")
print(f"  P('The' | DET) = {tagger.get_emission_prob('DET', 'The'):.4f}")
print(f"  P('barks' | VERB) = {tagger.get_emission_prob('VERB', 'barks'):.4f}")

EXAMPLE PROBABILITIES
Initial Probabilities (first word in sentence):
  P(ADJ) = 0.0500
  P(ADP) = 0.0500
  P(DET) = 0.7000
  P(NOUN) = 0.1500
  P(VERB) = 0.0500
\nSome Transition Probabilities:
  P(NOUN | DET) = 0.6000
  P(VERB | NOUN) = 0.7273
  P(ADJ | DET) = 0.2500
\nSome Emission Probabilities:
  P('dog' | NOUN) = 0.2162
  P('The' | DET) = 0.2571
  P('barks' | VERB) = 0.1143
